# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: a2228eac-de6b-431f-a691-f966f4a30eb7
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session a2228eac-de6b-431f-a691-f966f4a30eb7 to get into ready status...
Session a2228eac-de6b-431f-a691-f966f4a30eb7 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [4]:
df1 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("mode", "PERMISSIVE") \
    .csv("s3://aroma-heaven/dirty_cafe_sales.csv" )
df1.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|  Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_4977031|    Cake|       4|           3.0|       12.0|          Cash|In-store|      2023-05-16|
|   TXN_4271903|  Cookie|       4|           1.0|      ERROR|   Credit Card|In-store|      2023-07-19|
|   TXN_7034554|   Salad|       2|           5.0|       10.0|       UNKNOWN| UNKNOWN|      2023-04-27|
|   TXN_3160411|  Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
|   TXN_2602893|Smoothie|       5|           4.0|       20.0|   Credit Card|    NULL|      2023-03-31|
|   TXN_4433211| UNKNOWN|       3|           3.0|        9.0|         ERR

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim
df1 = df1.select([when(trim(col(c)) == "", None).otherwise(col(c)).alias(c) for c in df1.columns])
df_filled = df1.fillna("N/A")
df_filled.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|  Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_4977031|    Cake|       4|           3.0|       12.0|          Cash|In-store|      2023-05-16|
|   TXN_4271903|  Cookie|       4|           1.0|      ERROR|   Credit Card|In-store|      2023-07-19|
|   TXN_7034554|   Salad|       2|           5.0|       10.0|       UNKNOWN| UNKNOWN|      2023-04-27|
|   TXN_3160411|  Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
|   TXN_2602893|Smoothie|       5|           4.0|       20.0|   Credit Card|     N/A|      2023-03-31|
|   TXN_4433211| UNKNOWN|       3|           3.0|        9.0|         ERR

In [10]:
df_clean = df_filled.dropDuplicates()


In [11]:
df_clean.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_8078640|   Juice|       4|           3.0|       12.0|Digital Wallet|In-store|      2023-11-03|
|   TXN_9487821|   ERROR|       1|           5.0|        5.0|Digital Wallet|Takeaway|      2023-05-24|
|   TXN_5072031|    Cake|       2|           3.0|        6.0|          Cash|Takeaway|      2023-03-23|
|   TXN_9250710|   Salad|       1|           5.0|        5.0|          Cash|In-store|      2023-05-28|
|   TXN_8377564|    Cake|       5|           3.0|       15.0|           N/A|In-store|      2023-07-14|
|   TXN_7077119|   ERROR|       1|           4.0|        4.0|Digital Wallet|In-store|      2023-12-30|
|   TXN_9238066|     N/A|       5|           1.5|        7.5|Digital Wall

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, count, regexp_replace, to_date, month, year
df_clean = df_clean.withColumn("Quantity", col("Quantity").cast("int")) \
                   .withColumn("Price Per Unit", col("Price Per Unit").cast("double")) \
                   .withColumn("Total Spent", col("Total Spent").cast("double")) \
                   .withColumn("Transaction Date", to_date(col("Transaction Date"), "yyyy-MM-dd"))
df_clean.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_8078640|   Juice|       4|           3.0|       12.0|Digital Wallet|In-store|      2023-11-03|
|   TXN_9487821|   ERROR|       1|           5.0|        5.0|Digital Wallet|Takeaway|      2023-05-24|
|   TXN_5072031|    Cake|       2|           3.0|        6.0|          Cash|Takeaway|      2023-03-23|
|   TXN_9250710|   Salad|       1|           5.0|        5.0|          Cash|In-store|      2023-05-28|
|   TXN_8377564|    Cake|       5|           3.0|       15.0|           N/A|In-store|      2023-07-14|
|   TXN_7077119|   ERROR|       1|           4.0|        4.0|Digital Wallet|In-store|      2023-12-30|
|   TXN_9238066|     N/A|       5|           1.5|        7.5|Digital Wall

In [15]:
df_with_total = df_clean.withColumn("Computed_Total", col("Quantity") * col("Price Per Unit"))
df_with_total.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|Computed_Total|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+
|   TXN_8078640|   Juice|       4|           3.0|       12.0|Digital Wallet|In-store|      2023-11-03|          12.0|
|   TXN_9487821|   ERROR|       1|           5.0|        5.0|Digital Wallet|Takeaway|      2023-05-24|           5.0|
|   TXN_5072031|    Cake|       2|           3.0|        6.0|          Cash|Takeaway|      2023-03-23|           6.0|
|   TXN_9250710|   Salad|       1|           5.0|        5.0|          Cash|In-store|      2023-05-28|           5.0|
|   TXN_8377564|    Cake|       5|           3.0|       15.0|           N/A|In-store|      2023-07-14|          15.0|
|   TXN_7077119|   ERROR|       1|           4.0|       

In [17]:
df_filtered = df_with_total.filter(col("Total Spent") > 10)
df_filtered.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|Computed_Total|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+
|   TXN_8078640|   Juice|       4|           3.0|       12.0|Digital Wallet|In-store|      2023-11-03|          12.0|
|   TXN_8377564|    Cake|       5|           3.0|       15.0|           N/A|In-store|      2023-07-14|          15.0|
|   TXN_1967565|Smoothie|       5|           4.0|       20.0|   Credit Card|Takeaway|      2023-03-31|          20.0|
|   TXN_5043774|Sandwich|       3|           4.0|       12.0|          Cash|     N/A|      2023-06-08|          12.0|
|   TXN_1726858|Smoothie|       3|           4.0|       12.0|          Cash|In-store|            NULL|          12.0|
|   TXN_4134594|Sandwich|       3|           4.0|       

In [18]:
sales_summary = df_filtered.groupBy("Location").agg(
    sum("Total Spent").alias("Total_Revenue"),
    avg("Total Spent").alias("Average_Spend"),
    count("*").alias("Total_Transactions")
)
sales_summary.show()

+--------+-------------+------------------+------------------+
|Location|Total_Revenue|     Average_Spend|Total_Transactions|
+--------+-------------+------------------+------------------+
|In-store|      15766.0| 16.25360824742268|               970|
|     N/A|      16862.0| 16.32333010648596|              1033|
|Takeaway|      14867.0|16.194989106753813|               918|
| UNKNOWN|       1370.0| 16.91358024691358|                81|
|   ERROR|       1912.0|15.933333333333334|               120|
+--------+-------------+------------------+------------------+


In [20]:
df_time = df_filtered.withColumn("Month", month(col("Transaction Date"))) \
                     .withColumn("Year", year(col("Transaction Date")))
df_time.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+-----+----+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|Computed_Total|Month|Year|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+-----+----+
|   TXN_8078640|   Juice|       4|           3.0|       12.0|Digital Wallet|In-store|      2023-11-03|          12.0|   11|2023|
|   TXN_8377564|    Cake|       5|           3.0|       15.0|           N/A|In-store|      2023-07-14|          15.0|    7|2023|
|   TXN_1967565|Smoothie|       5|           4.0|       20.0|   Credit Card|Takeaway|      2023-03-31|          20.0|    3|2023|
|   TXN_5043774|Sandwich|       3|           4.0|       12.0|          Cash|     N/A|      2023-06-08|          12.0|    6|2023|
|   TXN_1726858|Smoothie|       3|           4.0|       12.0|          Cash|In-store|            

In [21]:
monthly_summary = df_time.groupBy("Year", "Month").agg(sum("Total Spent").alias("Monthly_Revenue"))
print("=== Monthly Revenue Summary ===")
monthly_summary.show()

=== Monthly Revenue Summary ===
+----+-----+---------------+
|Year|Month|Monthly_Revenue|
+----+-----+---------------+
|2023|   10|         4100.0|
|2023|    6|         4387.0|
|2023|    1|         3992.0|
|2023|    9|         3786.0|
|2023|    2|         3743.0|
|2023|   11|         4018.0|
|2023|    5|         4080.0|
|2023|   12|         4091.0|
|NULL| NULL|         2525.0|
|2023|    7|         3931.0|
|2023|    8|         4044.0|
|2023|    4|         4212.0|
|2023|    3|         3868.0|
+----+-----+---------------+


In [22]:
df_final = df_time.withColumn("Item", regexp_replace(col("Item"), "[^a-zA-Z0-9 ]", ""))
output_path = "cleaned_cafe_sales_output"
df_final.write.mode("overwrite").option("header", True).csv("s3://aroma-heaven/dirty_cafe_sales_output.csv")
df_final.show(5)

spark.stop()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+-----+----+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|Computed_Total|Month|Year|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+--------------+-----+----+
|   TXN_8078640|   Juice|       4|           3.0|       12.0|Digital Wallet|In-store|      2023-11-03|          12.0|   11|2023|
|   TXN_8377564|    Cake|       5|           3.0|       15.0|           N/A|In-store|      2023-07-14|          15.0|    7|2023|
|   TXN_1967565|Smoothie|       5|           4.0|       20.0|   Credit Card|Takeaway|      2023-03-31|          20.0|    3|2023|
|   TXN_5043774|Sandwich|       3|           4.0|       12.0|          Cash|     N/A|      2023-06-08|          12.0|    6|2023|
|   TXN_1726858|Smoothie|       3|           4.0|       12.0|          Cash|In-store|            